In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

In [2]:
class EmbeddingDataset(Dataset):
    """Wraps a .pt file that contains 'embeddings', 'attention_mask', 'labels'."""
 
    def __init__(self, path: str):
        data = torch.load(path, weights_only=True)
        self.embeddings    = data["embeddings"]      # (N, seq_len, 768)
        self.attention_mask = data["attention_mask"] # (N, seq_len)
        self.labels        = data["labels"]          # (N,)
 
    def __len__(self):
        return len(self.labels)
 
    def __getitem__(self, idx):
        return {
            "embeddings":    self.embeddings[idx],
            "attention_mask": self.attention_mask[idx],
            "label":         self.labels[idx],
        }

In [3]:
def mean_pool(embeddings: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    """
    Compute the masked mean of token embeddings.
 
    Args:
        embeddings:    (B, seq_len, hidden)
        attention_mask:(B, seq_len)  — 1 for real tokens, 0 for padding
 
    Returns:
        pooled: (B, hidden)
    """
    mask = attention_mask.unsqueeze(-1).float()          # (B, seq_len, 1)
    summed = (embeddings * mask).sum(dim=1)              # (B, hidden)
    counts = mask.sum(dim=1).clamp(min=1e-9)             # (B, 1)
    return summed / counts    

In [4]:
class LinearProbe(nn.Module):
    """Single linear layer on top of mean-pooled BERT embeddings."""
 
    def __init__(self, hidden_size: int = 768, num_classes: int = 77):
        super().__init__()
        self.classifier = nn.Linear(hidden_size, num_classes)
 
    def forward(self, embeddings: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        pooled = mean_pool(embeddings, attention_mask)   # (B, 768)
        return self.classifier(pooled)    

In [5]:
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: str) -> tuple[float, float]:
    """Returns (avg_loss, accuracy)."""
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, total = 0.0, 0, 0
 
    for batch in loader:
        emb   = batch["embeddings"].to(device)
        mask  = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
 
        logits = model(emb, mask)
        loss   = criterion(logits, labels)
 
        total_loss += loss.item() * labels.size(0)
        correct    += (logits.argmax(dim=-1) == labels).sum().item()
        total      += labels.size(0)
 
    return total_loss / total, correct / total

In [14]:
def train(
    train_path: str = r"D:/@FYP-IntentClassifier/text_preprocessing/train_embeddings.pt",
    val_path:   str = r"D:/@FYP-IntentClassifier/text_preprocessing/val_embeddings.pt",
    test_path:  str = r"D:/@FYP-IntentClassifier/text_preprocessing/test_embeddings.pt",
    num_epochs: int = 20,
    batch_size: int = 64,
    lr:         float = 1e-2,
    weight_decay: float = 1e-4,
    device:     str | None = None,
):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}\n")
 
    # ── Data ───────────────────────────────────────────────────────────────────
    train_ds = EmbeddingDataset(train_path)
    val_ds   = EmbeddingDataset(val_path)
    test_ds  = EmbeddingDataset(test_path)
 
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)
 
    print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}\n")
 
    # ── Model ──────────────────────────────────────────────────────────────────
    model = LinearProbe(hidden_size=768, num_classes=77).to(device)
    print(model)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Parameters: {total_params:,}\n")
 
    # ── Optimiser + scheduler ──────────────────────────────────────────────────
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = OneCycleLR(
        optimizer,
        max_lr=lr,
        steps_per_epoch=len(train_loader),
        epochs=num_epochs,
    )
 
    # ── Train ──────────────────────────────────────────────────────────────────
    best_val_acc = 0.0
 
    for epoch in range(1, num_epochs + 1):
        model.train()
        epoch_loss, epoch_correct, epoch_total = 0.0, 0, 0
 
        for batch in train_loader:
            emb    = batch["embeddings"].to(device)
            mask   = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
 
            optimizer.zero_grad()
            logits = model(emb, mask)
            loss   = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
 
            epoch_loss    += loss.item() * labels.size(0)
            epoch_correct += (logits.argmax(dim=-1) == labels).sum().item()
            epoch_total   += labels.size(0)
 
        train_loss = epoch_loss / epoch_total
        train_acc  = epoch_correct / epoch_total
 
        val_loss, val_acc = evaluate(model, val_loader, device)
 
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "best_linear_probe.pt")
 
        print(
            f"Epoch {epoch:02d}/{num_epochs} | "
            f"Train loss: {train_loss:.4f}  acc: {train_acc:.4f} | "
            f"Val loss: {val_loss:.4f}  acc: {val_acc:.4f}"
            + (" ← best" if val_acc >= best_val_acc else "")
        )
 
    # ── Test ───────────────────────────────────────────────────────────────────
    print("\nLoading best checkpoint for test evaluation…")
    model.load_state_dict(torch.load("best_linear_probe.pt", weights_only=True))
    test_loss, test_acc = evaluate(model, test_loader, device)
 
    print(f"\n{'='*55}")
    print(f"  Experiment 1 — Linear Probe (BERT baseline)")
    print(f"{'='*55}")
    print(f"  Best val  accuracy : {best_val_acc*100:.2f}%")
    print(f"  Test loss          : {test_loss:.4f}")
    print(f"  Test accuracy      : {test_acc*100:.2f}%")
    print(f"{'='*55}\n")
 
    return test_acc

In [15]:
if __name__ == "__main__":
    train()

Device: cuda

Train: 8495 | Val: 1498 | Test: 3076

LinearProbe(
  (classifier): Linear(in_features=768, out_features=77, bias=True)
)
Parameters: 59,213

Epoch 01/20 | Train loss: 3.7650  acc: 0.2764 | Val loss: 3.0096  acc: 0.5053 ← best
Epoch 02/20 | Train loss: 2.1979  acc: 0.6274 | Val loss: 1.4962  acc: 0.7076 ← best
Epoch 03/20 | Train loss: 1.1264  acc: 0.7746 | Val loss: 0.8846  acc: 0.7924 ← best
Epoch 04/20 | Train loss: 0.7153  acc: 0.8381 | Val loss: 0.6912  acc: 0.8284 ← best
Epoch 05/20 | Train loss: 0.5245  acc: 0.8747 | Val loss: 0.5525  acc: 0.8565 ← best
Epoch 06/20 | Train loss: 0.4066  acc: 0.8966 | Val loss: 0.5452  acc: 0.8505
Epoch 07/20 | Train loss: 0.3192  acc: 0.9225 | Val loss: 0.5091  acc: 0.8565 ← best
Epoch 08/20 | Train loss: 0.2627  acc: 0.9336 | Val loss: 0.5224  acc: 0.8558
Epoch 09/20 | Train loss: 0.2057  acc: 0.9520 | Val loss: 0.4724  acc: 0.8725 ← best
Epoch 10/20 | Train loss: 0.1630  acc: 0.9656 | Val loss: 0.4897  acc: 0.8578
Epoch 11/20 | Tr